# view_device.ipynb — release‑aware inspection (no writes)

Purpose: verify the **actual** shape of `device.csv` for a given CERT release, so the ETL can be built safely.
- Zero side effects: this notebook **does not** write Parquet or JSON outputs.
- Cheap peeks only: header reads, tiny samples, DuckDB virtual views.
- Confirms column presence (e.g., `file_tree` is missing in r3.1 but present in later releases), null rates, and key distributions.
- Validates that LDAP as‑of inputs exist for the intended join fidelity (lean vs full), **without** performing the join.

Use this before implementing or re‑running the `build_device` pipeline.


In [ ]:
# ========== 0) Bootstrap & imports (read-only) ==========
# Try package-style import first, then fall back to plain module.
import sys
from pathlib import Path

for cand in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (cand/"notebooks/nb_paths.py").exists():
        sys.path.insert(0, str(cand))
        break

from notebooks.nb_paths import bootstrap, csv_path, read_csv, iter_csv_chunks
from datetime import datetime
import pandas as pd

# Optional light deps; if missing, cells that use them will degrade gracefully.
try:
    import duckdb
except Exception:
    duckdb = None

env = bootstrap()  # resolves PROJECT, RAW, OUT, RELEASE from release.txt
print(f"RELEASE={env.RELEASE}\nRAW={env.RAW}\nOUT={env.OUT}")
DEVICE_CSV = csv_path(env, "device")
print("device.csv →", DEVICE_CSV, "exists:", DEVICE_CSV.exists())


## 1) Header‑only peek (no data load)
This reads a handful of bytes just to extract the header row, so you can see the real columns **without** slurping the file.


In [ ]:
import io, csv

def header_only(path: Path):
    with open(path, "rb") as f:
        head = f.read(8192)  # small buffer
    first = head.splitlines()[0].decode("utf-8", errors="replace")
    cols = next(csv.reader(io.StringIO(first)))
    return [c.strip() for c in cols]

cols = header_only(DEVICE_CSV)
print("Column count:", len(cols))
print(cols)


## 2) Presence map for release‑variant columns
Quick booleans for fragile fields. Add more as needed.


In [ ]:
presence = {
    "file_tree": "file_tree" in cols,
    "activity": "activity" in cols,
    "user"    : "user" in cols,
    "pc"      : "pc" in cols,
    "date"    : "date" in cols,
    "id"      : "id" in cols,
}
presence


## 3) Tiny sample to sanity‑check parsing
Load a **very small** sample to avoid memory pressure, then inspect dtypes and a few rows.


In [ ]:
sample_n = 5000
sample = read_csv(env, "device", nrows=sample_n)
print("sample rows:", len(sample), "| columns:", list(sample.columns)[:12], "...")
sample.head(5)


In [ ]:
# dtype snapshot (strings expected; ETL will coerce later)
sample.dtypes.to_frame("dtype").head(20)


## 4) Virtual view with DuckDB (optional, zero‑copy scans)
If DuckDB is available, create a read-only view and run a few cheap queries without loading everything into RAM.


In [ ]:
if duckdb is None:
    print("duckdb not available; skipping view queries.")
else:
    con = duckdb.connect()
    # SAMPLE_SIZE keeps auto-inference cheap; adjust upward if columns mis-infer.
    sql = "CREATE OR REPLACE VIEW v_device AS SELECT * FROM read_csv_auto('{}', SAMPLE_SIZE=20000);".format(str(DEVICE_CSV).replace("'", "''"))
    con.execute(sql)
    print("describe:")
    display(con.execute("DESCRIBE v_device;").fetchdf())
    print("row count estimate (approx):")
    display(con.execute("SELECT COUNT(*)::BIGINT AS n FROM v_device LIMIT 1;").fetchdf())
    print("top activities:")
    try:
        display(con.execute("SELECT activity, COUNT(*) AS n FROM v_device GROUP BY 1 ORDER BY n DESC NULLS LAST LIMIT 20;").fetchdf())
    except Exception as e:
        print("no 'activity' column or query failed:", e)


## 5) Timestamp sanity
Try to parse `date` to a timestamp and compute daily counts on a **limited** slice.


In [ ]:
from src.helpers.time import add_timestamp_and_month, month_start

probe = sample.copy()
if "date" in probe.columns:
    probe = add_timestamp_and_month(probe, "date")
else:
    probe["timestamp"] = pd.NaT
    probe["event_month"] = pd.NaT

probe["timestamp"] = pd.to_datetime(probe["timestamp"], errors="coerce")
probe["event_month"] = month_start(probe["event_month"])

print("parsed timestamps:", probe["timestamp"].notna().sum(), "/", len(probe))
probe[["timestamp","event_month"]].head(8)


In [ ]:
# Day histogram on the probe
if probe["timestamp"].notna().any():
    tmp = probe.dropna(subset=["timestamp"]).copy()
    tmp["day"] = tmp["timestamp"].dt.floor("D")
    counts = tmp.groupby("day").size().rename("n").reset_index()
    counts.head(15)
else:
    print("no parseable timestamps in probe sample.")


## 6) Null rates for key fields
Focus on a few columns that the ETL will depend on.


In [ ]:
key_cols = [c for c in ["user","pc","activity","date","id"] if c in sample.columns]
null_rate = sample[key_cols].isna().mean().sort_values(ascending=False).to_frame("null_rate")
null_rate


## 7) Cardinalities
How many distinct users and PCs in the probe set.


In [ ]:
from src.helpers.users import normalize_user_series
probe2 = sample.copy()
if "user" in probe2.columns:
    probe2["user_key"] = normalize_user_series(probe2["user"])
else:
    probe2["user_key"] = pd.NA

card = {
    "users_in_probe": int(probe2["user_key"].nunique(dropna=True)),
    "pcs_in_probe": int(probe2["pc"].nunique(dropna=True)) if "pc" in probe2.columns else 0
}
card


## 8) Join preflight
Confirm that the LDAP as‑of parquet exists at the fidelity you intend to use during ETL.

- For **lean** downstream: `out/<release>/ldap_v3/ldap_asof_by_month.parquet` (lean family usually fine for device/http).
- For **full** downstream: use the full family if you expect to keep more org context in the audit view.

This cell *does not* perform the join.


In [ ]:
from src.helpers.io import out_path
from pathlib import Path as _Path

LDAP_LEAN = out_path(env, "ldap_v3_lean", "ldap_asof_by_month")
LDAP_FULL = out_path(env, "ldap_v3_full", "ldap_asof_by_month")

exists = {
    "ldap_v3_lean": _Path(LDAP_LEAN).exists(),
    "ldap_v3_full": _Path(LDAP_FULL).exists(),
}
print(exists)

# minimal schema peek if present
for fam, p in [("ldap_v3_lean", LDAP_LEAN), ("ldap_v3_full", LDAP_FULL)]:
    if _Path(p).exists():
        df = pd.read_parquet(p, columns=["user_key","event_month","role"])
        print(f"{fam}: rows={len(df):,}  cols={list(df.columns)}")


## 9) Shared PC / Assigned PC helpers (optional)
If the logon pipeline produced these helper parquets, confirm they load and look sane. If missing, skip silently.


In [ ]:
# === 9) Shared PC / Assigned PC helpers (read-only, no mkdirs) ===
from pathlib import Path
from notebooks.nb_paths import bootstrap
import pandas as pd

env = bootstrap()
base = Path(env.OUT) / env.RELEASE

# Helpers are written by LOGON under the family root
shared_p   = base / "logon_v3" / "shared_pcs.parquet"
assigned_p = base / "logon_v3" / "assigned_pc.parquet"

helpers = {
    "shared_pcs": str(shared_p),
    "assigned_pc": str(assigned_p),
    "shared_exists": shared_p.exists(),
    "assigned_exists": assigned_p.exists(),
}
print(helpers)

if helpers["shared_exists"]:
    sp = pd.read_parquet(shared_p)
    display(sp.head(5))
if helpers["assigned_exists"]:
    ap = pd.read_parquet(assigned_p)
    display(ap.head(5))

## 10) Expected output contracts (for the ETL)
These are **assertions** about what your *final* device artifacts should look like when you implement or refactor the ETL:
- Exactly two outputs under `out/<release>/device_v3/`:
  - `device_v3_full/device_full.parquet`
  - `device_v3_lean/device_lean.parquet`
- Event‑level flags only in v3 (no counts/aggregates):
  - `after_hours_device`, `on_shared_pc`, `on_unassigned_pc` (or `on_non_assigned_pc` if you keep that name), `user_is_active_employee` derived post‑join.

This cell does not write anything; it’s here so you can copy the lists into your builder.


In [ ]:
# device outputs: two artifacts; include file_tree if present in this release
EXPECTED_FULL = [
    # event
    "timestamp","event_month","user_key","user_raw","pc","file_tree","activity",
    # org context (enriched)
    "email","role","is_admin","team","supervisor_key","last_seen",
    "employee_name","business_unit","functional_unit","department","supervisor",
    # event-level flags
    "after_hours","on_shared_pc","on_unassigned_pc","user_is_active_employee",
    # traceability
    "id","date",
]

EXPECTED_LEAN = [
    "timestamp","event_month","user_key","pc","file_tree","activity",
    "role","is_admin","supervisor_key","team",
    "after_hours","on_shared_pc","on_unassigned_pc","user_is_active_employee",
]

print("EXPECTED_FULL columns (suggested):", EXPECTED_FULL)
print("EXPECTED_LEAN columns (suggested):", EXPECTED_LEAN)

## 11) Cross‑release shape check (optional)
Run this cell with a list of releases to compare which columns exist per release. No data loading beyond header peeks.


In [ ]:
import io, csv
from pathlib import Path

def header_only(path: Path):
    with open(path, "rb") as f:
        head = f.read(8192)  # small buffer
    first = head.splitlines()[0].decode("utf-8", errors="replace")
    cols = next(csv.reader(io.StringIO(first)))
    return [c.strip() for c in cols]

def compare_headers(releases=("r3.1","r5.1")):
    out = {}
    here = Path.cwd()
    # try repo root if we're inside notebooks/
    if here.name == "notebooks":
        here = here.parent
    for rel in releases:
        path = here/"data"/rel/"device.csv"
        try:
            c = header_only(path)
            out[rel] = set(c)
        except Exception:
            out[rel] = set()
    all_cols = sorted(set().union(*out.values())) if out else []
    rows = []
    for col in all_cols:
        row = {"column": col}
        for rel in releases:
            row[rel] = col in out.get(rel, set())
        rows.append(row)
    import pandas as pd
    return pd.DataFrame(rows).set_index("column").sort_index()

compare_headers()


In [ ]:
# Build DEVICE (writes both device_v3_full/device_full.parquet and device_v3_lean/device_lean.parquet)
from scripts.etl import build_device

# Make sure LOGON (and its helpers) finished rebuilding first.
build_device(
    family="device_v3",                 # one family; function writes both subfolders
    ldap_family_for_join="ldap_v3_full",# use FULL LDAP to keep org context rich
    logon_family_for_pc="logon_v3",     # shared_pcs + assigned_pc live here
    overwrite=True,
)

In [ ]:
from pathlib import Path
import json
from notebooks.nb_paths import bootstrap

env = bootstrap()
qc_dir = Path(env.OUT) / env.RELEASE / "qc"
meta_path = qc_dir / "device_final_meta.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    display(meta)  # shows rows, columns, and relative paths for full/lean
    # quick file_tree presence check:
    dev_full_cols = set(meta["artifacts"]["device_full"]["cols"])
    dev_lean_cols = set(meta["artifacts"]["device_lean"]["cols"])
    print("file_tree in FULL:", "file_tree" in dev_full_cols)
    print("file_tree in LEAN:", "file_tree" in dev_lean_cols)
else:
    print("No device_final_meta.json yet — build_device hasn’t run or QC path differs.")

In [ ]:
import pyarrow.parquet as pq
from src.helpers.io import out_path

full_p = out_path(env, "device_v3", "device_v3_full/device_full")
lean_p = out_path(env, "device_v3", "device_v3_lean/device_lean")

for name, p in [("FULL", full_p), ("LEAN", lean_p)]:
    if Path(p).exists():
        print(name, "→", p)
        print(pq.ParquetFile(p).schema.names)
    else:
        print(name, "missing at", p)

In [ ]:
import pandas as pd
if Path(full_p).exists():
    df = pd.read_parquet(full_p, columns=[c for c in ["activity","after_hours","on_shared_pc","on_unassigned_pc","file_tree"] if c in pq.ParquetFile(full_p).schema.names])
    print("unique activity (sample):", sorted(df["activity"].dropna().astype(str).str.lower().unique()[:10]) if "activity" in df.columns else "(no activity column)")
    for c in ["after_hours","on_shared_pc","on_unassigned_pc"]:
        if c in df.columns:
            print(c, "→", df[c].mean(skipna=True))
    if "file_tree" in df.columns:
        print("file_tree non-null rate:", 1 - df["file_tree"].isna().mean())